# Etapa 9 — Entrenamiento final (Tesla T4)
Los dos modelos siguen desde su `best.pt` de la etapa 8 por 100 épocas más, con el mismo dataset (China + Japón, India, República Checa y Estados Unidos) y los mismos hiperparámetros (`configs/stage9_final.yaml`). En la etapa 8 las curvas seguían subiendo en la época 50.

Orden: celdas 1 a 5, **SMOKE**, y recién después el entrenamiento. Si Colab se desconecta, vuelva a ejecutar las celdas 1 a 5 y luego la celda **Reanudar** del modelo que se cortó.

La celda **TEST** va al final y se ejecuta **una sola vez**, cuando se decida que estos son los modelos finales.

In [ ]:
# 1. Comprobar GPU asignada
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Seleccione Runtime > Change runtime type > T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
assert 'T4' in torch.cuda.get_device_name(0).upper()

In [ ]:
# 2. Montar Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Extraer el paquete en almacenamiento rápido temporal
from pathlib import Path
import shutil, zipfile
ZIP_PATH = Path('/content/drive/MyDrive/TP1_LMD/stage9/packages/colab_stage6.zip')
PACKAGE_ROOT = Path('/content/colab_stage6')
RESULTS_ROOT = Path('/content/drive/MyDrive/TP1_LMD/stage9/results')
CONFIG = PACKAGE_ROOT / 'configs' / 'stage9_final.yaml'
STAGE8 = Path('/content/drive/MyDrive/TP1_LMD/stage8/results')
assert ZIP_PATH.is_file(), ZIP_PATH
assert (STAGE8 / 'dataset_stage8_manifest.json').is_file(), 'Falta el dataset de la etapa 8 en Drive'
for name in ('lmd_yolo', 'yolov8n'):
    assert (STAGE8 / 'training' / name / 'weights' / 'best.pt').is_file(), f'Falta el best.pt de la etapa 8 de {name}'
if PACKAGE_ROOT.exists():
    shutil.rmtree(PACKAGE_ROOT)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall('/content')
assert CONFIG.is_file()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print(PACKAGE_ROOT, RESULTS_ROOT, CONFIG)

In [ ]:
# 4. Verificar todos los hashes del paquete
import subprocess, sys
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/manifests.py'), 'verify-package'], check=True)

In [ ]:
# 5. Instalar dependencias generales; NO instala ultralytics ni reemplaza torch CUDA
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PACKAGE_ROOT/'requirements-colab.txt')], check=True)

In [ ]:
# 6. SMOKE: 1 época sobre 2 % de TRAIN para cada modelo
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage9.py'), 'smoke', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)
import json
print(json.dumps(json.loads((RESULTS_ROOT/'smoke_gate.json').read_text()), indent=2))
print('STOP: revise el smoke antes de entrenar.')

## Gate manual para las 100 épocas
Cambie la variable a `True` solo después de revisar el smoke.

In [ ]:
RUN_LONG_TRAINING = False  # cambiar manualmente a True tras revisar el smoke
assert RUN_LONG_TRAINING, 'Gate cerrado: no se lanzará el entrenamiento automáticamente.'

In [ ]:
# 7A. LMD_YOLO: 100 épocas desde el best.pt de la etapa 8
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage9.py'), 'train', '--model', 'lmd_yolo', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

In [ ]:
# 7A-R. Reanudar LMD_YOLO si Colab se desconectó (solo en ese caso)
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage9.py'), 'train', '--model', 'lmd_yolo', '--resume', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

In [ ]:
# 7B. YOLOv8n: 100 épocas desde el best.pt de la etapa 8 (referencia del paper)
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage9.py'), 'train', '--model', 'yolov8n', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

In [ ]:
# 7B-R. Reanudar YOLOv8n si Colab se desconectó (solo en ese caso)
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage9.py'), 'train', '--model', 'yolov8n', '--resume', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

In [ ]:
# 8. Comparación etapa 8 vs etapa 9 y curvas (requiere los dos modelos); no usa TEST
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage9.py'), 'compare', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

## TEST final (una sola vez)
Evalúa los dos modelos finales sobre el TEST de China y el test bloqueado de los otros países. Solo se ejecuta cuando se haya decidido que estos son los modelos finales; el script se niega a evaluarlo una segunda vez.

In [ ]:
EVALUAR_TEST_FINAL = False  # cambiar manualmente a True solo cuando estos sean los modelos finales
assert EVALUAR_TEST_FINAL, 'Gate cerrado: el TEST no se evalúa todavía.'
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage9.py'), 'test', '--confirmo-modelo-final', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)